In [1]:
import sys
print("Python:", sys.executable)

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
print("API key:", f"{api_key[:10]}..."if api_key else "NOT LOADED")

import anthropic
client = anthropic.Anthropic()
print("Client created successfully")

Python: C:\Users\BWaters\AppData\Local\anaconda3\envs\agents\python.exe
API key: sk-ant-api...
Client created successfully


In [2]:
api_key = os.getenv("ANTHROPIC_API_KEY")
print("API key:", f"{api_key[:10]}..."if api_key else "NOT LOADED")

import anthropic
client = anthropic.Anthropic()
print("Client created successfully")

API key: sk-ant-api...
Client created successfully


In [3]:
def calculator(operation, a, b):
    """Perform basic arithmetic operations."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b if b != 0 else "Error: division by zero"
    else:
        return f"Error: unknown operation '{operation}'"

# Quick test — this is just a Python function, nothing Claude-related yet
print(calculator("multiply", 47, 83))

3901


In [4]:
tools = [
    {
        "name": "calculator",
        "description": "Perform basic arithmetic operations: add, subtract, multiply, divide.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform."
                },
                "a": {
                    "type": "number",
                    "description": "The first number."
                },
                "b": {
                    "type": "number",
                    "description": "The second number."
                }
            },
            "required": ["operation", "a", "b"]
        }
    }
]

In [5]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is 47 times 83?"}
    ]
)

print("Stop reason:", response.stop_reason)
print()
print("Response content:")
for block in response.content:
    print(block)

Stop reason: tool_use

Response content:
ToolUseBlock(id='toolu_01Nwk9sHmzeULwL5aSbCtjtR', caller=DirectCaller(type='direct'), input={'operation': 'multiply', 'a': 47, 'b': 83}, name='calculator', type='tool_use')


In [6]:
# Find the tool use block in the response
tool_use = next(block for block in response.content if block.type == "tool_use")

print(f"Claude wants to call: {tool_use.name}")
print(f"With arguments: {tool_use.input}")
print()

# Run the actual Python function with those arguments
tool_result = calculator(**tool_use.input)
print(f"Tool result: {tool_result}")
print()

# Now send the result back to Claude as the next message
messages = [
    {"role": "user", "content": "What is 47 times 83?"},
    {"role": "assistant", "content": response.content},  # Claude's tool-use request
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": str(tool_result)
            }
        ]
    }
]

final_response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=tools,
    messages=messages
)

print("Final answer:", final_response.content[0].text)

Claude wants to call: calculator
With arguments: {'operation': 'multiply', 'a': 47, 'b': 83}

Tool result: 3901

Final answer: 47 times 83 equals **3,901**.


In [7]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is 17 plus 29, and then multiply that by 4?"}
    ]
)

print("Stop reason:", response.stop_reason)
print()
print("Response content:")
for block in response.content:
    print(block)

Stop reason: tool_use

Response content:
TextBlock(citations=None, text="I'll help you calculate that step by step.\n\nFirst, let me add 17 plus 29:", type='text')
ToolUseBlock(id='toolu_01Pbuyq9FPCa3u5QmAMqBPX9', caller=DirectCaller(type='direct'), input={'operation': 'add', 'a': 17, 'b': 29}, name='calculator', type='tool_use')


In [8]:
# Start the conversation
messages = [
    {"role": "user", "content": "What is 17 plus 29, and then multiply that by 4?"}
]

# Keep looping until Claude stops asking for tools
while True:
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    print(f"Stop reason: {response.stop_reason}")
    
    # Add Claude's response to the conversation
    messages.append({"role": "assistant", "content": response.content})
    
    # If Claude isn't asking for a tool, we're done
    if response.stop_reason != "tool_use":
        # Print the final text answer
        for block in response.content:
            if block.type == "text":
                print(f"\nFinal answer: {block.text}")
        break
    
    # Otherwise, find the tool call(s) and execute them
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            print(f"  Claude wants to call {block.name} with {block.input}")
            result = calculator(**block.input)
            print(f"  Result: {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result)
            })
    
    # Send all the tool results back in one message
    messages.append({"role": "user", "content": tool_results})

Stop reason: tool_use
  Claude wants to call calculator with {'operation': 'add', 'a': 17, 'b': 29}
  Result: 46
Stop reason: tool_use
  Claude wants to call calculator with {'operation': 'multiply', 'a': 46, 'b': 4}
  Result: 184
Stop reason: end_turn

Final answer: The answer is **184**.

To break it down:
- 17 + 29 = 46
- 46 × 4 = 184


In [9]:
## Adding a second tool: get_stock_price

In [10]:
import yfinance as yf

def get_stock_price(ticker):
    """Fetch the most recent closing price for a given stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="5d")
        if data.empty:
            return f"Error: No data found for ticker '{ticker}'"
        latest_close = data["Close"].iloc[-1]
        latest_date = data.index[-1].date()
        return f"{ticker} closed at ${latest_close:.2f} on {latest_date}"
    except Exception as e:
        return f"Error fetching {ticker}: {str(e)}"

# Quick test — pure Python, no Claude yet
print(get_stock_price("AAPL"))

AAPL closed at $270.76 on 2026-04-24


In [11]:
import yfinance as yf

def get_stock_price(ticker):
    """Fetch the most recent price for a given stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="5d")
        if data.empty:
            return f"Error: No data found for ticker '{ticker}'"
        latest_price = data["Close"].iloc[-1]
        latest_date = data.index[-1].date()
        return f"{ticker} most recent price: ${latest_price:.2f} (as of {latest_date})"
    except Exception as e:
        return f"Error fetching {ticker}: {str(e)}"

# Quick test — pure Python, no Claude yet
print(get_stock_price("AAPL"))

AAPL most recent price: $270.76 (as of 2026-04-24)


In [12]:
tools = [
    {
        "name": "calculator",
        "description": "Perform basic arithmetic operations: add, subtract, multiply, divide.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform."
                },
                "a": {"type": "number", "description": "The first number."},
                "b": {"type": "number", "description": "The second number."}
            },
            "required": ["operation", "a", "b"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Fetch the most recent closing price for a given stock ticker (e.g., AAPL, MSFT, NVDA). Returns the price and date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol (e.g., 'AAPL' for Apple)."
                }
            },
            "required": ["ticker"]
        }
    }
]

In [13]:
def execute_tool(name, inputs):
    """Dispatch a tool call to the right Python function."""
    if name == "calculator":
        return calculator(**inputs)
    elif name == "get_stock_price":
        return get_stock_price(**inputs)
    else:
        return f"Error: unknown tool '{name}'"

def run_agent(user_message):
    """Run the agent loop until Claude finishes responding."""
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        print(f"Stop reason: {response.stop_reason}")
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    print(f"\nFinal answer: {block.text}")
            return
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  Calling {block.name} with {block.input}")
                result = execute_tool(block.name, block.input)
                print(f"  Result: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        messages.append({"role": "user", "content": tool_results})

In [14]:
run_agent("What did AAPL, MSFT, and NVDA close at, and what's the average of those three prices?")

Stop reason: tool_use
  Calling get_stock_price with {'ticker': 'AAPL'}
  Result: AAPL most recent price: $270.76 (as of 2026-04-24)
  Calling get_stock_price with {'ticker': 'MSFT'}
  Result: MSFT most recent price: $419.14 (as of 2026-04-24)
  Calling get_stock_price with {'ticker': 'NVDA'}
  Result: NVDA most recent price: $209.49 (as of 2026-04-24)
Stop reason: tool_use
  Calling calculator with {'operation': 'add', 'a': 270.76, 'b': 419.14}
  Result: 689.9
Stop reason: tool_use
  Calling calculator with {'operation': 'add', 'a': 689.9, 'b': 209.49}
  Result: 899.39
Stop reason: tool_use
  Calling calculator with {'operation': 'divide', 'a': 899.39, 'b': 3}
  Result: 299.7966666666667
Stop reason: end_turn

Final answer: Here are the closing prices as of April 24, 2026:

- **AAPL**: $270.76
- **MSFT**: $419.14
- **NVDA**: $209.49

**Average**: $299.80 (rounded to 2 decimal places)


In [15]:
run_agent("If I buy 100 shares each of AAPL, MSFT, and NVDA, what's the total cost?")

Stop reason: tool_use
  Calling get_stock_price with {'ticker': 'AAPL'}
  Result: AAPL most recent price: $270.97 (as of 2026-04-24)
  Calling get_stock_price with {'ticker': 'MSFT'}
  Result: MSFT most recent price: $419.15 (as of 2026-04-24)
  Calling get_stock_price with {'ticker': 'NVDA'}
  Result: NVDA most recent price: $209.32 (as of 2026-04-24)
Stop reason: tool_use
  Calling calculator with {'operation': 'multiply', 'a': 270.97, 'b': 100}
  Result: 27097.000000000004
  Calling calculator with {'operation': 'multiply', 'a': 419.15, 'b': 100}
  Result: 41915.0
  Calling calculator with {'operation': 'multiply', 'a': 209.32, 'b': 100}
  Result: 20932.0
Stop reason: tool_use
  Calling calculator with {'operation': 'add', 'a': 27097, 'b': 41915}
  Result: 69012
Stop reason: tool_use
  Calling calculator with {'operation': 'add', 'a': 69012, 'b': 20932}
  Result: 89944
Stop reason: end_turn

Final answer: Based on the most recent prices (as of April 24, 2026):

- **AAPL**: $270.97 ×

In [16]:
run_agent("I want to compare the 5-year return of AAPL vs. the S&P 500 (ticker SPY). What do you need from me to answer that?")

Stop reason: end_turn

Final answer: To calculate the 5-year return for AAPL vs SPY, I can fetch the current prices for both using the available tools. However, I would need the stock prices from 5 years ago to calculate the actual returns.

Here's what I need from you:

1. **Historical prices from 5 years ago**: The stock prices for AAPL and SPY from approximately 5 years ago (same date in 2020)

OR

If you'd like, I can:
- Fetch the **current prices** for both AAPL and SPY right now, and then you could provide me with the historical prices from 5 years ago
- Calculate the percentage return once we have both data points

The 5-year return formula would be: **((Current Price - Price 5 Years Ago) / Price 5 Years Ago) × 100%**

Would you like me to fetch the current prices first, and then you can provide the historical prices? Or do you already have the historical price data from 5 years ago?


In [17]:
def get_historical_return(ticker, years):
    """Compute the total return of a stock over N years."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period=f"{years + 1}y")
        if data.empty or len(data) < 2:
            return f"Error: Insufficient data for {ticker}"
        
        start_price = data["Close"].iloc[0]
        end_price = data["Close"].iloc[-1]
        total_return = ((end_price - start_price) / start_price) * 100
        
        start_date = data.index[0].date()
        end_date = data.index[-1].date()
        
        return f"{ticker} return from {start_date} to {end_date}: {total_return:.2f}% (start ${start_price:.2f}, end ${end_price:.2f})"
    except Exception as e:
        return f"Error computing {ticker} return: {str(e)}"

# Quick test — pure Python, no Claude
print(get_historical_return("AAPL", 5))

AAPL return from 2020-04-24 to 2026-04-24: 296.44% (start $68.37, end $271.06)


In [18]:
tools = [
    {
        "name": "calculator",
        "description": "Perform basic arithmetic operations: add, subtract, multiply, divide.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform."
                },
                "a": {"type": "number", "description": "The first number."},
                "b": {"type": "number", "description": "The second number."}
            },
            "required": ["operation", "a", "b"]
        }
    },
    {
        "name": "get_stock_price",
        "description": "Fetch the most recent closing price for a given stock ticker (e.g., AAPL, MSFT, NVDA).",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol (e.g., 'AAPL' for Apple)."
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_historical_return",
        "description": "Compute the total percentage return of a stock over a specified number of years. Returns start price, end price, dates, and percentage return.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol (e.g., 'AAPL' for Apple, 'SPY' for S&P 500 ETF)."
                },
                "years": {
                    "type": "integer",
                    "description": "The number of years over which to compute the return."
                }
            },
            "required": ["ticker", "years"]
        }
    }
]

print(f"Tools registered: {[t['name'] for t in tools]}")

Tools registered: ['calculator', 'get_stock_price', 'get_historical_return']


In [19]:
def execute_tool(name, inputs):
    """Dispatch a tool call to the right Python function."""
    if name == "calculator":
        return calculator(**inputs)
    elif name == "get_stock_price":
        return get_stock_price(**inputs)
    elif name == "get_historical_return":
        return get_historical_return(**inputs)
    else:
        return f"Error: unknown tool '{name}'"

In [20]:
run_agent("Compare the 5-year return of AAPL vs. the S&P 500 (ticker SPY). Which performed better, and by how much?")

Stop reason: tool_use
  Calling get_historical_return with {'ticker': 'AAPL', 'years': 5}
  Result: AAPL return from 2020-04-24 to 2026-04-24: 296.66% (start $68.37, end $271.21)
  Calling get_historical_return with {'ticker': 'SPY', 'years': 5}
  Result: SPY return from 2020-04-24 to 2026-04-24: 173.75% (start $260.12, end $712.07)
Stop reason: tool_use
  Calling calculator with {'operation': 'subtract', 'a': 296.66, 'b': 173.75}
  Result: 122.91000000000003
Stop reason: end_turn

Final answer: ## Results

**AAPL (Apple) performed significantly better than the S&P 500 over the past 5 years:**

- **AAPL**: +296.66% return ($68.37 → $271.21)
- **SPY (S&P 500)**: +173.75% return ($260.12 → $712.07)

**Difference**: AAPL outperformed the S&P 500 by **122.91 percentage points**.

This means that if you had invested $10,000 in each:
- AAPL would have grown to ~$39,666
- SPY would have grown to ~$27,375

Apple's exceptional performance over this period significantly exceeded the broader mark

In [21]:
run_agent("Which had higher 5-year returns: AAPL, MSFT, NVDA, or SPY? Rank them and tell me the spread between best and worst.")

Stop reason: tool_use
  Calling get_historical_return with {'ticker': 'AAPL', 'years': 5}
  Result: AAPL return from 2020-04-24 to 2026-04-24: 296.19% (start $68.37, end $270.89)
  Calling get_historical_return with {'ticker': 'MSFT', 'years': 5}
  Result: MSFT return from 2020-04-24 to 2026-04-24: 153.02% (start $165.81, end $419.52)
  Calling get_historical_return with {'ticker': 'NVDA', 'years': 5}
  Result: NVDA return from 2020-04-24 to 2026-04-24: 2786.09% (start $7.21, end $208.13)
  Calling get_historical_return with {'ticker': 'SPY', 'years': 5}
  Result: SPY return from 2020-04-24 to 2026-04-24: 173.61% (start $260.12, end $711.71)
Stop reason: tool_use
  Calling calculator with {'operation': 'subtract', 'a': 2786.09, 'b': 153.02}
  Result: 2633.07
Stop reason: end_turn

Final answer: ## 5-Year Returns Ranking (2020-2024):

1. **NVDA: 2,786.09%** 🥇
2. **AAPL: 296.19%** 🥈
3. **SPY: 173.61%** 🥉
4. **MSFT: 153.02%**

**Spread between best and worst: 2,633.07 percentage points**
